# 00 · Quickstart: geodesic distances on crossing lines

**Question this notebook answers:** what does the full pipeline do on the simplest intersecting structure, two lines crossing in 2D, and how does it compare to standard graph-based geodesics?

Pipeline: points → localized Tyler covariances → determinant-penalized edge weights on a kNN graph → Dijkstra.

Run with *Restart & Run All*. All parameters are in the first code cell and everything is seeded.

In [ ]:
# --- Parameters -----------------------------------------------------------------------------
SEED     = 0
N_POINTS = 1500
NOISE    = 0.005    # Gaussian noise (std) added to the points
N_LINES  = 2

TYLER_K  = 30       # neighborhood size of the Tyler estimator
GRAPH_K  = 15       # neighbors in the shortest-path graph
ALPHA    = 2.0      # determinant-penalty exponent (0 disables the penalty)
BETA     = 3.0      # edge-length exponent (> 1: Fermat-style path selection)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Ellipse
from scipy.sparse.csgraph import dijkstra

import CovBIM as cb

print("available datasets:", cb.datasets.available())

## 1. Data

`crossing_lines` returns noise-free segments crossing at the origin; `make` then adds the Gaussian noise (seeded). Labels give the branch of each point and are used only for plotting and diagnostics, never by the method.

In [ ]:
cloud = cb.datasets.make("crossing_lines", n_points=N_POINTS, noise=NOISE, seed=SEED, n_lines=N_LINES)
print(cloud.points.shape, "branches:", np.unique(cloud.labels))

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(*cloud.points.T, c=cloud.labels, cmap="tab10", s=4)
ax.set_aspect("equal")
ax.set_title("crossing_lines (colored by branch)")
plt.show()

# Source point: the end of the first line, far from the crossing.
mask0 = cloud.labels == 0
source = int(np.flatnonzero(mask0)[np.argmin(cloud.points[mask0, 0])])
print("source index:", source)

## 2. Fit the metric

`fit` estimates the covariances, builds the weighted graph and checks connectivity. After that, `distances_from` is cheap, so changing the source costs almost nothing.

In [ ]:
config = cb.PipelineConfig.from_dict({
    "covariance": {"k": TYLER_K},
    "metric": {"alpha": ALPHA, "beta": BETA},
    "graph": {"k": GRAPH_K},
})
gm = cb.GeodesicMetric(config).fit(cloud.points)

print(f"Tyler converged at {gm.tyler_.converged.mean():.1%} of the points "
      f"(median iterations: {np.median(gm.tyler_.iterations):.0f})")
print("connected components of the graph:", gm.n_components_)

## 3. Inspect the intermediate quantities

Left: local covariance ellipses (a random subset of points). Along a branch they should be elongated in the line direction; near the crossing they become rounder.
Right: the determinant of the trace-normalized covariance, the scalar field that penalizes paths near intersections.

In [ ]:
rng = np.random.default_rng(SEED)
subset = rng.choice(len(cloud), size=150, replace=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))

ax1.scatter(*cloud.points.T, s=2, c="lightgray")
for i in subset:
    w, V = np.linalg.eigh(gm.covariances[i])          # ascending eigenvalues
    angle = np.degrees(np.arctan2(V[1, 1], V[0, 1]))  # direction of the main axis
    r = gm.tyler_.radii[i]
    ax1.add_patch(Ellipse(gm.tyler_.locations[i], 2 * r * np.sqrt(w[1]), 2 * r * np.sqrt(w[0]),
                          angle=angle, fill=False, color="C3", lw=0.8))
ax1.set_aspect("equal")
ax1.set_title("local covariance ellipses")

cb.viz.plot_field(cloud.points, gm.determinants, ax=ax2, cmap="magma", title="det of normalized covariance")
plt.show()

## 4. Geodesic distances from the source

Same source, three metrics. The color scale is set per panel (the power-weighted distance does not have the same units as the others).

In [ ]:
d = gm.distances_from(source)   # beta-selected paths, re-measured with beta = 1 (Riemannian length)

d_euc = cb.baselines.BASELINES["euclidean"](cloud.points, source, k=GRAPH_K)
d_fer = cb.baselines.BASELINES["fermat"](cloud.points, source, k=GRAPH_K, p=BETA)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, dist) in zip(axes, [("ours", d), ("Euclidean kNN graph", d_euc), (f"power-weighted (p={BETA:g})", d_fer)]):
    cb.viz.plot_distance_field(cloud.points, dist, source=source, ax=ax, title=name)
plt.tight_layout()
plt.show()

**What to look for:** how the distance to points of the *other* branch behaves close to the crossing, and whether it is smooth along each branch. Try `ALPHA = 0` in the parameter cell to see the effect of the determinant penalty alone.

### Isolevels

One panel per level shows only the points at a given distance from the source (a thin band), which makes differences between metrics easier to read than a color map.

In [ ]:
finite = d[np.isfinite(d)]
levels = np.quantile(finite, [0.2, 0.4, 0.6, 0.8])
width = 0.02 * np.ptp(finite)

fig = cb.viz.plot_distance_isolevels(cloud.points, d, source, levels, width, ncols=4)
plt.show()

## 5. Sanity checks

1. The C++ Dijkstra must agree with scipy's Dijkstra on the same graph.
2. Median distance per branch, normalized by the largest finite distance of each method, to compare the methods on the same scale.

In [ ]:
# 1. Cross-check with scipy (raw beta-weighted distances, no beta = 1 recomputation)
d_scipy = dijkstra(gm.csr_graph(), directed=False, indices=source)
d_raw = gm.distances_from(source, recompute=False)
print("C++ Dijkstra matches scipy:", np.allclose(d_scipy, d_raw))

# 2. Median normalized distance per branch
for name, dist in [("ours", d), ("euclidean", d_euc), ("power-weighted", d_fer)]:
    scale = dist[np.isfinite(dist)].max()
    medians = {int(l): round(float(np.median(dist[cloud.labels == l] / scale)), 3)
               for l in np.unique(cloud.labels)}
    print(f"{name:>15}: {medians}")

## Next steps

- Change the dataset: `cb.datasets.make("sphere_plane", ...)` or `"plane_with_pole"`, and plot in 3D (`plot_field` handles 2D and 3D).
- Increase `NOISE` or add outliers with `ambient=0.1` in `make(...)`.
- Compare `spd_metric` values (`"frobenius"`, `"log_euclidean"`, `"airm"`) in the `metric` config.
- Anything worth keeping goes to `tests/` (as an assertion) or `experiments/paper_figures/` (as a script with a config), not into this notebook.